# Feature engineering improvement search

Two-phase Optuna hyperparameter search for XGBoost:

1. **Phase 1** — all 12 original raw features, no pruning/engineering.
2. **Phase 2** — drop confirmed-noise features (`age`, `gender`, `stress_level`, `academic_work_impact`), add the 4 engineered features that showed real permutation importance (`social_share`, `gaming_share`, `study_share`, `unaccounted_screen_time`).

Each phase gets its own wall-clock time budget (set via `MAX_SECONDS_PHASE1`/`MAX_SECONDS_PHASE2`). Set `QUICK_TEST = True` to run both phases for ~2 minutes locally just to confirm everything works before kicking off the full 3h/3h run on Kaggle.

Uses Optuna (TPE sampler + median pruning) instead of plain random search: it reuses information from earlier trials to bias later ones, and pruning kills clearly bad trials early — both help more within a hard time budget than blind random sampling.

In [4]:
import os
import time
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

try:
    import optuna
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna"], check=True)
    import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [5]:
# ===== Config =====
SEED = 42
N_SPLITS = 3
EARLY_STOPPING_ROUNDS = 50

QUICK_TEST = True  # set False for the real Kaggle run
MAX_SECONDS_PHASE1 = 120 if QUICK_TEST else 3 * 3600
MAX_SECONDS_PHASE2 = 120 if QUICK_TEST else 3 * 3600

print(f"QUICK_TEST={QUICK_TEST} | phase1 budget={MAX_SECONDS_PHASE1}s | phase2 budget={MAX_SECONDS_PHASE2}s")

QUICK_TEST=True | phase1 budget=120s | phase2 budget=120s


In [6]:
# ===== Data path auto-detect (local vs Kaggle) =====
local_train = "/workspaces/Kaggle/KagglePredictPhoneAddiction/data/train.csv"
kaggle_train = "/kaggle/input/competitions/playground-series-s6e8/train.csv"
train_path = local_train if os.path.exists(local_train) else kaggle_train
if not os.path.exists(train_path):
    raise FileNotFoundError("Could not find train.csv in local or Kaggle path.")

print("Train file:", train_path)
df = pd.read_csv(train_path)

out_dir = "/kaggle/working/feature_eng_improvement_outputs" if os.path.exists("/kaggle/working") else "/workspaces/Kaggle/KagglePredictPhoneAddiction/feature_eng_improvement_outputs"
os.makedirs(out_dir, exist_ok=True)
print("Output dir:", out_dir)

y = df["addicted_label"].astype(int)

Train file: /workspaces/Kaggle/KagglePredictPhoneAddiction/data/train.csv
Output dir: /workspaces/Kaggle/KagglePredictPhoneAddiction/feature_eng_improvement_outputs


In [7]:
# ===== Phase 1: all original raw features =====
numeric_cols = [
    "age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
    "work_study_hours", "sleep_hours", "notifications_per_day",
    "app_opens_per_day", "weekend_screen_time",
]
categorical_cols = ["gender", "stress_level", "academic_work_impact"]

feature_cols_phase1 = numeric_cols + categorical_cols
X_phase1 = df[feature_cols_phase1].copy()
for col in categorical_cols:
    X_phase1[col] = X_phase1[col].astype("category")

print("Phase 1 features:", feature_cols_phase1)

Phase 1 features: ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'gender', 'stress_level', 'academic_work_impact']


In [ ]:
# ===== Phase 2: drop confirmed-noise features, add engineered ones =====
def add_engineered_features(source_df):
    eng = pd.DataFrame(index=source_df.index)
    # ratios can divide by 0 -> inf; treat those as missing rather than a real signal
    with np.errstate(divide="ignore", invalid="ignore"):
        eng["social_share"] = source_df["social_media_hours"] / source_df["daily_screen_time_hours"]
        eng["gaming_share"] = source_df["gaming_hours"] / source_df["daily_screen_time_hours"]
        eng["study_share"] = source_df["work_study_hours"] / source_df["daily_screen_time_hours"]
    eng["unaccounted_screen_time"] = source_df["daily_screen_time_hours"] - (
        source_df["social_media_hours"] + source_df["gaming_hours"] + source_df["work_study_hours"]
    )
    return eng.replace([np.inf, -np.inf], np.nan)

# age, gender, stress_level, academic_work_impact confirmed noise via gain + permutation importance in 01_eda.ipynb
pruned_numeric_cols = [c for c in numeric_cols if c != "age"]
categorical_cols_phase2 = []  # all categoricals were confirmed noise, none kept

eng_features = add_engineered_features(df)
X_phase2 = pd.concat([df[pruned_numeric_cols], eng_features], axis=1)
feature_cols_phase2 = list(X_phase2.columns)

print("Phase 2 features:", feature_cols_phase2)

Phase 2 features: ['daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'social_share', 'gaming_share', 'study_share', 'unaccounted_screen_time']


In [9]:
# ===== Search machinery (shared by both phases) =====
def make_objective(X, y, categorical_cols, seed, n_splits):
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 300, 3000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "min_child_weight": trial.suggest_float("min_child_weight", 1, 12),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 100.0, log=True),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        }

        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        fold_scores = []

        for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
            X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
            y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

            common = dict(
                **params,
                tree_method="hist",
                enable_categorical=len(categorical_cols) > 0,
                eval_metric="auc",
                early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                random_state=seed,
                n_jobs=-1,
            )
            try:
                model = XGBClassifier(device="cuda", **common)
                model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
            except Exception:
                model = XGBClassifier(device="cpu", **common)
                model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

            pred = model.predict_proba(X_va)[:, 1]
            fold_scores.append(roc_auc_score(y_va, pred))

            trial.report(float(np.mean(fold_scores)), step=fold_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()

        return float(np.mean(fold_scores))
    return objective


def run_search(X, y, categorical_cols, max_seconds, out_prefix, seed=SEED, n_splits=N_SPLITS):
    csv_path = os.path.join(out_dir, f"{out_prefix}_results.csv")
    json_path = os.path.join(out_dir, f"{out_prefix}_best.json")

    def save_callback(study, _trial):
        rows = [
            {"trial": t.number, "value": t.value, "state": str(t.state), **t.params}
            for t in study.trials if t.value is not None
        ]
        if rows:
            pd.DataFrame(rows).sort_values("value", ascending=False).to_csv(csv_path, index=False)
        if study.best_trial is not None:
            with open(json_path, "w", encoding="utf-8") as f:
                json.dump({"best_value": study.best_value, "best_params": study.best_params}, f, indent=2)

    sampler = optuna.samplers.TPESampler(seed=seed)
    pruner = optuna.pruners.MedianPruner(n_warmup_steps=1)
    study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)

    print(f"Starting search '{out_prefix}' | time budget: {max_seconds/3600:.2f}h ({max_seconds:.0f}s)")
    start = time.perf_counter()
    study.optimize(
        make_objective(X, y, categorical_cols, seed, n_splits),
        timeout=max_seconds,
        callbacks=[save_callback],
        catch=(Exception,),
    )
    elapsed = time.perf_counter() - start
    save_callback(study, None)

    print(f"'{out_prefix}' done | trials={len(study.trials)} | best AUC={study.best_value:.4f} | elapsed={elapsed/60:.1f} min")
    print("Best params:", study.best_params)
    print("Saved results CSV:", csv_path)
    print("Saved best JSON:", json_path)
    return study

In [10]:
study_phase1 = run_search(X_phase1, y, categorical_cols, MAX_SECONDS_PHASE1, "phase1_all_features")

Starting search 'phase1_all_features' | time budget: 0.03h (120s)


/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [21:34:04] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [21:34:04] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [21:35:10] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [21:35:10] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round =

'phase1_all_features' done | trials=1 | best AUC=0.9633 | elapsed=2.9 min
Best params: {'n_estimators': 1311, 'learning_rate': 0.13125830316209655, 'max_depth': 8, 'min_child_weight': 7.585243326167403, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 0.00019517224641449495, 'reg_lambda': 21.42302175774105, 'gamma': 3.005575058716044}
Saved results CSV: /workspaces/Kaggle/KagglePredictPhoneAddiction/feature_eng_improvement_outputs/phase1_all_features_results.csv
Saved best JSON: /workspaces/Kaggle/KagglePredictPhoneAddiction/feature_eng_improvement_outputs/phase1_all_features_best.json


In [11]:
study_phase2 = run_search(X_phase2, y, categorical_cols_phase2, MAX_SECONDS_PHASE2, "phase2_pruned_engineered")

Starting search 'phase2_pruned_engineered' | time budget: 0.03h (120s)


/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [21:36:58] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [21:36:58] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [21:37:54] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [21:37:54] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round =

'phase2_pruned_engineered' done | trials=1 | best AUC=0.9638 | elapsed=2.2 min
Best params: {'n_estimators': 1311, 'learning_rate': 0.13125830316209655, 'max_depth': 8, 'min_child_weight': 7.585243326167403, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 0.00019517224641449495, 'reg_lambda': 21.42302175774105, 'gamma': 3.005575058716044}
Saved results CSV: /workspaces/Kaggle/KagglePredictPhoneAddiction/feature_eng_improvement_outputs/phase2_pruned_engineered_results.csv
Saved best JSON: /workspaces/Kaggle/KagglePredictPhoneAddiction/feature_eng_improvement_outputs/phase2_pruned_engineered_best.json


In [12]:
print("=== Comparison ===")
print(f"Phase 1 (all original features)       best CV AUC: {study_phase1.best_value:.4f}  ({len(study_phase1.trials)} trials)")
print(f"Phase 2 (pruned + engineered features) best CV AUC: {study_phase2.best_value:.4f}  ({len(study_phase2.trials)} trials)")

=== Comparison ===
Phase 1 (all original features)       best CV AUC: 0.9633  (1 trials)
Phase 2 (pruned + engineered features) best CV AUC: 0.9638  (1 trials)
